# Content Creator Workflow Test

This notebook tests the Content Creator workflow:
1. Create Content Creator user
2. Assign CONTENT_CREATOR role
3. Content Creator creates a note (draft)
4. Submit note for admin review
5. Admin approves
6. Note published to repository
7. Teacher searches and copies note

In [ ]:
import requests
import json
from uuid import uuid4

# Service URLs
AUTH_URL = "http://localhost:8081"
TENANT_URL = "http://localhost:8082"
ROLE_PERM_URL = "http://localhost:8080"
NOTES_URL = "http://localhost:8088"
WORKFLOW_URL = "http://localhost:8086"

# Test tenant (Euroschool)
TENANT_ID = None
TENANT_KEY = "euroschool"

# Users
content_creator = {"name": "Suresh", "email": "suresh@euroschool.com", "password": "Password123!"}
admin = {"name": "Admin", "email": "admin@euroschool.com", "password": "Password123!"}
teacher = {"name": "Priya", "email": "priya@euroschool.com", "password": "Password123!"}

# Tokens storage
tokens = {}
user_ids = {}
role_ids = {}

def headers(token=None, tenant_id=None, user_id=None):
    h = {"Content-Type": "application/json"}
    if token:
        h["Authorization"] = f"Bearer {token}"
    if tenant_id:
        h["X-Tenant-Id"] = str(tenant_id)
    if user_id:
        h["X-User-Id"] = str(user_id)
    return h

def print_response(resp, label="Response"):
    print(f"\n{label}: {resp.status_code}")
    try:
        print(json.dumps(resp.json(), indent=2))
    except:
        print(resp.text[:500] if resp.text else "(empty)")

print("Setup complete!")

## Step 1: Get or Create Tenant

In [ ]:
# Try to resolve existing tenant first
resp = requests.get(f"{TENANT_URL}/v1/resolve", params={"tenantKey": TENANT_KEY})

if resp.status_code == 200:
    TENANT_ID = resp.json().get("tenantId")
    print(f"Found existing tenant: {TENANT_ID}")
else:
    # Create tenant
    payload = {
        "tenantKey": TENANT_KEY,
        "name": "Euroschool"
    }
    resp = requests.post(f"{TENANT_URL}/v1/tenants", json=payload, headers=headers())
    print_response(resp, "Create Tenant")
    if resp.status_code in [200, 201]:
        TENANT_ID = resp.json().get("id")
        print(f"Created tenant: {TENANT_ID}")

print(f"\nUsing TENANT_ID: {TENANT_ID}")

## Step 2: Create Roles (CONTENT_CREATOR, ADMIN, TEACHER)

In [ ]:
roles_to_create = [
    {"name": "CONTENT_CREATOR", "description": "Creates content for repository"},
    {"name": "TENANT_ADMIN", "description": "Admin who approves content"},
    {"name": "SUBJECT_TEACHER", "description": "Teacher who uses content"}
]

for role in roles_to_create:
    payload = {
        "name": role["name"],
        "description": role["description"],
        "active": True
    }
    resp = requests.post(
        f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/roles",
        json=payload,
        headers=headers(user_id="admin")
    )
    print_response(resp, f"Create Role: {role['name']}")
    if resp.status_code in [200, 201]:
        role_ids[role["name"]] = resp.json().get("id")

# If roles already exist, fetch them
resp = requests.get(f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/roles")
if resp.status_code == 200:
    for r in resp.json():
        role_ids[r["name"]] = r["id"]

print(f"\nRole IDs: {role_ids}")

## Step 3: Create Users (Content Creator, Admin, Teacher)

In [ ]:
users_to_create = [
    ("content_creator", content_creator),
    ("admin", admin),
    ("teacher", teacher)
]

for key, user in users_to_create:
    payload = {
        "tenantId": str(TENANT_ID),
        "email": user["email"],
        "password": user["password"],
        "name": user["name"],
        "joinMethod": "SELF_SIGNUP"
    }
    resp = requests.post(f"{AUTH_URL}/auth/signup", json=payload, headers=headers())
    print_response(resp, f"Signup: {user['name']}")
    if resp.status_code in [200, 201]:
        user_ids[key] = resp.json().get("userId")

print(f"\nUser IDs: {user_ids}")

## Step 4: Assign Roles to Users

In [ ]:
role_assignments = [
    ("content_creator", "CONTENT_CREATOR"),
    ("admin", "TENANT_ADMIN"),
    ("teacher", "SUBJECT_TEACHER")
]

for user_key, role_name in role_assignments:
    if user_key not in user_ids or role_name not in role_ids:
        print(f"Skipping {user_key} -> {role_name} (missing IDs)")
        continue
    
    payload = {
        "roleId": str(role_ids[role_name]),
        "scopeType": "TENANT",
        "scopeId": None,
        "status": "ACTIVE"
    }
    resp = requests.post(
        f"{ROLE_PERM_URL}/tenants/{TENANT_ID}/users/{user_ids[user_key]}/roles",
        json=payload,
        headers=headers(user_id="admin")
    )
    print_response(resp, f"Assign {role_name} to {user_key}")

print("\nRoles assigned!")

## Step 5: Login Users

In [ ]:
for key, user in users_to_create:
    payload = {
        "tenantId": str(TENANT_ID),
        "identifier": user["email"],
        "password": user["password"]
    }
    resp = requests.post(f"{AUTH_URL}/auth/login", json=payload, headers=headers())
    print_response(resp, f"Login: {user['name']}")
    if resp.status_code == 200:
        tokens[key] = resp.json().get("accessToken")

print(f"\nTokens acquired for: {list(tokens.keys())}")

## Step 6: Content Creator Creates a Note (Draft)

Suresh (Content Creator) creates a Photosynthesis note with proper tags.

In [ ]:
note_payload = {
    "title": "Photosynthesis - Complete Guide",
    "summary": "Comprehensive notes on photosynthesis for Grade 4 Science",
    "contentMd": """# Photosynthesis

Photosynthesis is the process by which plants make their food using sunlight.

## Key Equation

$$6CO_2 + 6H_2O + \\text{light energy} \\rightarrow C_6H_{12}O_6 + 6O_2$$

## Main Steps

1. **Light Absorption** - Chlorophyll absorbs sunlight
2. **Water Splitting** - Water molecules are split into hydrogen and oxygen
3. **Carbon Fixation** - CO2 is converted to glucose

## Factors Affecting Photosynthesis

- Light intensity
- Temperature
- CO2 concentration
""",
    "tags": [
        "#subject-science",
        "#topic-photosynthesis",
        "#grade-4",
        "#repository"  # Marks this for content repository
    ]
}

resp = requests.post(
    f"{NOTES_URL}/notes",
    json=note_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Create Note (Content Creator)")

note_id = None
note_version_id = None
if resp.status_code in [200, 201]:
    note_id = resp.json().get("id")
    note_version_id = resp.json().get("latestVersionId")
    print(f"\nNote ID: {note_id}")
    print(f"Version ID: {note_version_id}")

## Step 7: Content Creator Submits Note for Review

Creates a workflow and submits for admin approval.

In [ ]:
# Create workflow
workflow_payload = {
    "contentId": note_id,
    "contentVersionId": note_version_id,
    "titleSnapshot": "Photosynthesis - Complete Guide",
    "publishTargets": {
        "scope": "REPOSITORY",  # Publishing to repository, not a class
        "classIds": []
    }
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow",
    json=workflow_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Create Workflow")

workflow_id = None
if resp.status_code in [200, 201]:
    workflow_id = resp.json().get("workflowId") or resp.json().get("id")
    print(f"\nWorkflow ID: {workflow_id}")

In [ ]:
# Submit for review (assign admin as reviewer)
submit_payload = {
    "reviewerUserIds": [str(user_ids.get("admin"))],
    "requiredApprovals": 1,
    "note": "Please review this photosynthesis content for accuracy and syllabus alignment."
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/submit",
    json=submit_payload,
    headers=headers(
        token=tokens.get("content_creator"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("content_creator")
    )
)
print_response(resp, "Submit for Review")

review_task_id = None
if resp.status_code in [200, 201]:
    # Extract review task ID from response
    reviews = resp.json().get("reviews") or resp.json().get("reviewTasks") or []
    if reviews:
        review_task_id = reviews[0].get("id")
    print(f"\nReview Task ID: {review_task_id}")

## Step 8: Admin Reviews and Approves

Admin logs in and approves the content.

In [ ]:
# Get workflow status first
resp = requests.get(
    f"{WORKFLOW_URL}/workflow/{workflow_id}",
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID
    )
)
print_response(resp, "Workflow Status (Admin View)")

# If we didn't get review_task_id earlier, try to extract it now
if not review_task_id and resp.status_code == 200:
    workflow_data = resp.json()
    reviews = workflow_data.get("reviews") or workflow_data.get("reviewTasks") or []
    if reviews:
        review_task_id = reviews[0].get("id")
        print(f"\nFound Review Task ID: {review_task_id}")

In [ ]:
# Admin approves the review
approve_payload = {
    "comment": "Content is accurate and well-structured. Approved for repository."
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/reviews/{review_task_id}/approve",
    json=approve_payload,
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("admin")
    )
)
print_response(resp, "Admin Approve Review")

## Step 9: Publish to Repository

After approval, the workflow can be published to the content repository.

In [ ]:
# Publish the workflow (makes note available in repository)
publish_payload = {
    "publishAt": None,  # Publish immediately
    "publishTargets": {
        "scope": "REPOSITORY",
        "classIds": []
    }
}

resp = requests.post(
    f"{WORKFLOW_URL}/workflow/{workflow_id}/publish",
    json=publish_payload,
    headers=headers(
        token=tokens.get("admin"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("admin")
    )
)
print_response(resp, "Publish to Repository")

## Step 10: Teacher Searches Repository

Priya (Teacher) searches the content repository for photosynthesis notes.

In [ ]:
# Teacher searches for notes with #repository tag and #topic-photosynthesis
search_params = {
    "tag": "#topic-photosynthesis",
    "status": "PUBLISHED"
}

resp = requests.get(
    f"{NOTES_URL}/notes",
    params=search_params,
    headers=headers(
        token=tokens.get("teacher"),
        tenant_id=TENANT_ID,
        user_id=user_ids.get("teacher")
    )
)
print_response(resp, "Teacher Search Repository")

repository_notes = []
if resp.status_code == 200:
    repository_notes = resp.json() if isinstance(resp.json(), list) else resp.json().get("content", [])
    print(f"\nFound {len(repository_notes)} notes in repository")

## Step 11: Teacher Copies Note for Their Mind Map

Teacher creates a copy of the note for use in their mind map (copy-on-use model).

In [ ]:
# First, get the note details
if note_id:
    resp = requests.get(
        f"{NOTES_URL}/notes/{note_id}",
        headers=headers(
            token=tokens.get("teacher"),
            tenant_id=TENANT_ID,
            user_id=user_ids.get("teacher")
        )
    )
    print_response(resp, "Get Original Note")
    
    if resp.status_code == 200:
        original_note = resp.json()
        
        # Create a copy for teacher's use
        # This could be a dedicated API like POST /notes/{id}/copy
        # For now, we'll create a new note referencing the original
        copy_payload = {
            "title": original_note.get("title"),
            "summary": original_note.get("summary"),
            "contentMd": original_note.get("contentMd") or original_note.get("content"),
            "tags": [
                "#subject-science",
                "#topic-photosynthesis",
                "#grade-4",
                "#mindmap-biology-fundamentals",  # Links to teacher's mind map
                f"#copied-from-{note_id}"  # Reference to original
            ]
        }
        
        resp = requests.post(
            f"{NOTES_URL}/notes",
            json=copy_payload,
            headers=headers(
                token=tokens.get("teacher"),
                tenant_id=TENANT_ID,
                user_id=user_ids.get("teacher")
            )
        )
        print_response(resp, "Teacher Creates Copy")
        
        teacher_note_id = None
        if resp.status_code in [200, 201]:
            teacher_note_id = resp.json().get("id")
            print(f"\nTeacher's Note Copy ID: {teacher_note_id}")

## Summary: Content Creator Workflow

```
Content Creator (Suresh)
    |
    v
Creates Note (DRAFT)
    |
    v
Submits for Review -----> Admin (Reviewer)
    |                          |
    |                          v
    |                     Reviews & Approves
    |                          |
    v                          v
Note PUBLISHED to Repository <--
    |
    v
Teacher (Priya) Searches
    |
    v
Copies Note (own copy)
    |
    v
Links to Mind Map
```

In [ ]:
# Final verification: Check workflow state
if workflow_id:
    resp = requests.get(
        f"{WORKFLOW_URL}/workflow/{workflow_id}",
        headers=headers(
            token=tokens.get("admin"),
            tenant_id=TENANT_ID
        )
    )
    print_response(resp, "Final Workflow State")
    
    if resp.status_code == 200:
        state = resp.json().get("state")
        print(f"\n--- WORKFLOW STATE: {state} ---")

## Credits Tracking (Future Enhancement)

Content Creators have monthly credit limits. Each approved note consumes 1 credit.

```python
# Example credit check (to be implemented in content-workflow-service)
# GET /credits/{userId}/balance
# Response: { "remaining": 10, "used": 5, "limit": 15 }

# On successful publish:
# POST /credits/{userId}/deduct
# { "amount": 1, "reason": "NOTE_PUBLISHED", "contentId": "..." }
```